In [12]:
import pandas as pd
import pandasql
import json

In [ ]:
# 获取token
import requests
import os

login_url = "https://admin.summerfarm.net/authentication/auth/username/login"
login_data = {
    "username": "peng.tang@summerfarm.net",
    "password": os.getenv("XIANMU_ADMIN_PASSWORD"),
}

token = requests.post(login_url, data=login_data).json()

token_str = token.get("data").get("token")

print(token)


headers = {
    "token": token_str,
    "xm-rqid": "create_fake_merchant_tp",
    "xm-uid": "2047",
    "Content-Type": "application/json;charset=UTF-8",
}

print(headers)

## 创建人群&创建活动。

In [64]:
# 上传文件：

import requests

headers = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en;q=0.9,zh-CN;q=0.8,zh-TW;q=0.7,zh;q=0.6",
    "origin": "https://admin.summerfarm.net",
    "referer": "https://admin.summerfarm.net/summerfarm/home.html",
    "sec-ch-ua": '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "token": token_str,  # Dynamic token input
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "xm-biz": "xm-manage",
    "xm-page": "/activity/circle-create",
    "xm-phone": "18618107293",
    "xm-platform": "web",
    "xm-rqid": "1733986846730-23679621",
    "xm-uid": "2047",
}


def upload_file(file_path, creator=""):
    url = "https://admin.summerfarm.net/circle-pak/create"
    file_name = os.path.basename(file_path)
    file_name = file_name[0 : file_name.rfind(".")]
    form_data = {
        "id": "",
        "name": file_name,
        "remark": "唐鹏批量创建:" + file_name,
        "userNum": "",
        "createWay": "1",
        "creator": creator,
        "updateWay": "0",
        "status": "",
        "usingNum": "",
    }

    print(form_data)

    # File to upload
    files = {
        "file": (
            file_path,
            open(file_path, "rb"),
            "application/vnd.ms-excel",
        )
    }

    # Sending POST request
    response = requests.post(url, headers=headers, data=form_data, files=files)
    return response.json()

In [ ]:
import pandas as pd
import requests

df = pd.read_excel("./TOP50人群包ID.xlsx", sheet_name=0)

# group by spu_name and sku_id
grouped_df = df.groupby(["spu_name", "sku_id"])

# save each group's cust_id into a local xls file and upload to server
for name, group_df in grouped_df:
    spu_name, sku_id = name
    file_name = f"人群_{spu_name}_{sku_id}"
    file_path = f"{file_name}.xls"
    print(f"spu_name:{spu_name}, file_path:{file_path}")
    group_df["cust_id"].to_excel(
        file_path, index=False, header=["mId"], engine="openpyxl"
    )
    result = upload_file(file_path, file_name)
    print(f"{file_name}: {result}")

In [ ]:
def list_audience_by_creator_name(name: str = "唐鹏"):
    url = "https://admin.summerfarm.net/circle-pak/list"
    return requests.post(
        url, json={"pageNo": 1, "pageSize": 200, "creator": name}, headers=headers
    ).json()


audience = list_audience_by_creator_name().get("data").get("list")
audience_df=pd.DataFrame(audience)
audience_df

In [ ]:
audience_df = audience_df[audience_df["name"].str.contains("人群_")]
audience_df["sku"] = audience_df.apply(
    lambda row: row["name"].split("_")[-1] if len(row["name"].split("_")) > 2 else None,
    axis=1,
)
audience_df

## 创建活动

1. 活动生效日期：12月13日10点开始-12月19日晚22点结束
1. 活动名称：商品名称XXX召回1213，如安佳淡奶油召回1213

In [ ]:
activity_df = pd.read_excel("./TOP50特价格式.xlsx", sheet_name=0)
activity_df


# group by spu_name and sku_id
activity_grouped_df = activity_df.groupby(["sku"]).agg(
    {"活动价（默认指定价）": "first", "限购件数（为空则不限购）": "first"}
)

activity_grouped_df=activity_grouped_df.reset_index()

activity_grouped_df

In [ ]:
from datetime import datetime


def get_sku_detail(sku="814153876") -> dict:
    url = f"https://admin.summerfarm.net/inventory/selectLikeBySkuOrName/1/10?queryStr={sku}"
    return requests.get(url=url, headers=headers).json()["data"]["list"][0]


print(get_sku_detail())


def create_activity(
    name: str,
    scopeId: int,
    sku: str,
    start_time="2024-12-13 10:00",
    end_time="2024-12-19 22:00",
):
    url = "https://admin.summerfarm.net/activity/upsert/add/basic-info"
    sku_price_df = activity_grouped_df[activity_grouped_df["sku"] == sku]
    print(f"sku_price_df:{sku_price_df.to_json(force_ascii=False)}")
    sku_obj = get_sku_detail(sku=sku)
    sku_obj["accountLimit"] = 1
    limit_quantity = int(sku_price_df.iloc[0]["限购件数（为空则不限购）"])
    price = int(sku_price_df.iloc[0]["活动价（默认指定价）"])
    sku_obj["limitQuantity"] = f"{limit_quantity}"
    sku_obj["activityLadderConfigDTOList"] = [
        {"unit": 1, "roundingMode": 0, "adjustType": 0, "amount": price}
    ]

    data = {
        "basicInfoDTO": {
            "name": name.replace("人群_", ""),
            "isPermanent": 0,
            "startTime": start_time,
            "endTime": end_time,
            "type": 0,
            "tag": 2,
            "remark": f"唐鹏批量创建于:{datetime.now()}",
            "platform": 0,
        },
        "itemConfigDTO": {"skuDetailList": [sku_obj]},
        "scopeConfigDTO": {"scopeIds": [scopeId], "scopeType": 1},
    }
    print(data)
    return requests.post(url, json=data, headers=headers).json()


for _index, row in audience_df.iterrows():
    if row["sku"] is None:
        print(f"SKU有误:{row['sku']}, {row.to_dict()}")
        continue
    result = create_activity(name=row["name"], scopeId=row["id"], sku=row["sku"])
    print(result, row.to_dict())